# CLIP Embedding Space Analysis

This notebook analyzes CLIP's embedding space by generating text embeddings and visualizing them in 2D using PCA (Principal Component Analysis).

## 1. Install and Import Dependencies

In [ ]:
# Install required packages
!pip install torch torchvision ftfy regex transformers scikit-learn matplotlib numpy
!pip install git+https://github.com/openai/CLIP.git

In [ ]:
import clip
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CLIP available: {clip.available_models()}")

## 2. Load CLIP Model

In [ ]:
# Load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

print(f"Using device: {device}")
print(f"Model loaded: ViT-B/32")

## 3. Define Text Samples

We'll create diverse categories of text to see how CLIP organizes them in embedding space.

In [ ]:
# Define text samples across different categories
text_samples = {
    'Animals': [
        'a photo of a dog',
        'a photo of a cat',
        'a photo of a lion',
        'a photo of an elephant',
        'a photo of a bird',
        'a photo of a fish'
    ],
    'Vehicles': [
        'a photo of a car',
        'a photo of a bicycle',
        'a photo of a train',
        'a photo of an airplane',
        'a photo of a boat',
        'a photo of a motorcycle'
    ],
    'Food': [
        'a photo of pizza',
        'a photo of a burger',
        'a photo of sushi',
        'a photo of pasta',
        'a photo of salad',
        'a photo of ice cream'
    ],
    'Nature': [
        'a photo of mountains',
        'a photo of the ocean',
        'a photo of a forest',
        'a photo of a desert',
        'a photo of a waterfall',
        'a photo of flowers'
    ],
    'Objects': [
        'a photo of a computer',
        'a photo of a phone',
        'a photo of a book',
        'a photo of a chair',
        'a photo of a lamp',
        'a photo of a clock'
    ]
}

# Flatten the dictionary
all_texts = []
categories = []
labels = []

for category, texts in text_samples.items():
    for text in texts:
        all_texts.append(text)
        categories.append(category)
        labels.append(text.replace('a photo of ', ''))

print(f"Total text samples: {len(all_texts)}")
print(f"Categories: {list(text_samples.keys())}")

## 4. Generate CLIP Text Embeddings

In [ ]:
# Tokenize and encode text
text_tokens = clip.tokenize(all_texts).to(device)

# Generate embeddings
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    # Normalize embeddings
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# Convert to numpy
embeddings = text_features.cpu().numpy()

print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

## 5. Apply PCA for Dimensionality Reduction

We'll reduce the high-dimensional embeddings (512D for ViT-B/32) to 2D for visualization.

In [ ]:
# Apply PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

print(f"2D embedding shape: {embeddings_2d.shape}")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

## 6. Visualize Embeddings in 2D Space

In [ ]:
# Create visualization
plt.figure(figsize=(14, 10))

# Define colors for each category
colors = {
    'Animals': '#FF6B6B',
    'Vehicles': '#4ECDC4',
    'Food': '#FFE66D',
    'Nature': '#95E1D3',
    'Objects': '#C7CEEA'
}

# Plot each category
for category in text_samples.keys():
    mask = np.array(categories) == category
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[category],
        label=category,
        s=200,
        alpha=0.7,
        edgecolors='black',
        linewidth=1.5
    )

# Add labels for each point
for i, label in enumerate(labels):
    plt.annotate(
        label,
        (embeddings_2d[i, 0], embeddings_2d[i, 1]),
        xytext=(5, 5),
        textcoords='offset points',
        fontsize=8,
        alpha=0.8
    )

plt.xlabel('First Principal Component', fontsize=12)
plt.ylabel('Second Principal Component', fontsize=12)
plt.title('CLIP Text Embeddings Visualization (PCA)', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Analyze Similarity Matrix

Let's create a heatmap showing the cosine similarity between different text embeddings.

In [ ]:
# Compute similarity matrix
similarity_matrix = embeddings @ embeddings.T

# Create heatmap
plt.figure(figsize=(16, 14))
im = plt.imshow(similarity_matrix, cmap='coolwarm', vmin=0, vmax=1)

# Add colorbar
plt.colorbar(im, label='Cosine Similarity')

# Set ticks and labels
plt.xticks(range(len(labels)), labels, rotation=90, fontsize=8)
plt.yticks(range(len(labels)), labels, fontsize=8)

plt.title('CLIP Text Embedding Similarity Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Find Most Similar and Dissimilar Pairs

In [ ]:
# Find most similar pairs (excluding self-similarity)
similarity_no_diag = similarity_matrix.copy()
np.fill_diagonal(similarity_no_diag, -1)

# Most similar
most_similar_idx = np.unravel_index(similarity_no_diag.argmax(), similarity_no_diag.shape)
print("Most Similar Pair:")
print(f"  '{all_texts[most_similar_idx[0]]}' <-> '{all_texts[most_similar_idx[1]]}'")
print(f"  Similarity: {similarity_matrix[most_similar_idx]:.4f}")

# Most dissimilar
most_dissimilar_idx = np.unravel_index(similarity_no_diag.argmin(), similarity_no_diag.shape)
print("\nMost Dissimilar Pair:")
print(f"  '{all_texts[most_dissimilar_idx[0]]}' <-> '{all_texts[most_dissimilar_idx[1]]}'")
print(f"  Similarity: {similarity_matrix[most_dissimilar_idx]:.4f}")

# Top 5 most similar pairs
print("\nTop 5 Most Similar Pairs:")
flat_indices = np.argsort(similarity_no_diag.flatten())[::-1]
shown = set()
count = 0
for flat_idx in flat_indices:
    i, j = np.unravel_index(flat_idx, similarity_no_diag.shape)
    pair = tuple(sorted([i, j]))
    if pair not in shown:
        shown.add(pair)
        print(f"  {count+1}. '{all_texts[i]}' <-> '{all_texts[j]}' = {similarity_matrix[i, j]:.4f}")
        count += 1
        if count >= 5:
            break

## 9. Category Centroids Analysis

Analyze the average embedding for each category.

In [ ]:
# Compute centroids for each category
centroids = {}
centroids_2d = {}

for category in text_samples.keys():
    mask = np.array(categories) == category
    centroids[category] = embeddings[mask].mean(axis=0)
    centroids_2d[category] = embeddings_2d[mask].mean(axis=0)

# Plot categories with centroids
plt.figure(figsize=(14, 10))

for category in text_samples.keys():
    mask = np.array(categories) == category
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        c=colors[category],
        label=category,
        s=150,
        alpha=0.5,
        edgecolors='black',
        linewidth=1
    )
    
    # Plot centroid
    plt.scatter(
        centroids_2d[category][0],
        centroids_2d[category][1],
        c=colors[category],
        s=500,
        alpha=1.0,
        edgecolors='black',
        linewidth=3,
        marker='*'
    )
    
    # Label centroid
    plt.annotate(
        f'{category}\n(centroid)',
        centroids_2d[category],
        xytext=(10, 10),
        textcoords='offset points',
        fontsize=10,
        fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor=colors[category], alpha=0.7)
    )

plt.xlabel('First Principal Component', fontsize=12)
plt.ylabel('Second Principal Component', fontsize=12)
plt.title('CLIP Text Embeddings with Category Centroids', fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compute inter-centroid similarities
print("\nInter-Category Centroid Similarities:")
category_names = list(text_samples.keys())
for i, cat1 in enumerate(category_names):
    for cat2 in category_names[i+1:]:
        similarity = np.dot(centroids[cat1], centroids[cat2])
        print(f"  {cat1} <-> {cat2}: {similarity:.4f}")

## 10. Summary and Insights

Key observations from the analysis:

1. **Embedding Dimension**: CLIP ViT-B/32 produces 512-dimensional embeddings for text
2. **PCA Visualization**: The 2D projection helps visualize semantic relationships
3. **Category Clustering**: Similar concepts tend to cluster together in embedding space
4. **Cosine Similarity**: Measures how similar different text descriptions are
5. **Centroids**: Show the "average" representation of each category

The visualization demonstrates how CLIP learns to organize semantic concepts in a meaningful way, where similar items are closer together in the embedding space.